# Toy Test: YOLOv8-pose Pruning End-to-End

This notebook runs the simple pruning script on the tiny `coco8-pose` dataset (8 images, ~1 min on a single GPU) and verifies that:

1. Training completes without errors
2. The saved `last.pt` contains the actual pruned tensors (smaller params)
3. The `pruned_model.onnx` export reflects the real pruned architecture
4. Validation on the loaded model matches the auto-logged metrics

**Use this notebook as a smoke test** before kicking off a long `coco-pose` run, to verify your environment is set up correctly. Once it passes, swap `coco8-pose.yaml` → `coco-pose.yaml`, bump `--epochs` to 50, and launch the real run from the command line.

## 1. Environment check

Verify the four pinned dependencies are importable. If any fails, run `pip install -r requirements.txt` and restart the kernel.

In [ ]:
import torch
import ultralytics
import torch_pruning as tp
import fasterai

print(f'torch         {torch.__version__}')
print(f'ultralytics   {ultralytics.__version__}')
print(f'torch-pruning {tp.__version__ if hasattr(tp, "__version__") else "(no version attr)"}')
print(f'fasterai      {fasterai.__version__ if hasattr(fasterai, "__version__") else "(no version attr)"}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Run the simple pruning script

Tiny config: 4 epochs on coco8-pose at imgsz=320, batch=2. The pruning callback fires schedule-driven prune events; with `--asymmetric --asymmetric-scale 0.5` we target ~6% sparsity (gentle for a toy run).

Wall time: ~30–60 seconds on a modern GPU.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'prune_yolov8_pose_simple.py',
    '--data',  'coco8-pose.yaml',
    '--epochs', '4',
    '--imgsz',  '320',
    '--batch',  '2',
    '--lr',     '5e-5',
    '--asymmetric',
    '--asymmetric-scale', '0.5',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print('--- tail of stdout ---')
print('\n'.join(result.stdout.splitlines()[-30:]))
if result.returncode != 0:
    print('--- stderr ---'); print(result.stderr[-2000:])
    raise RuntimeError(f'script exited with code {result.returncode}')

## 3. Inspect the auto-logged experiment

The script appends a line to `experiments.jsonl`. Read the last entry and check final params/mAP.

In [ ]:
import json
from pathlib import Path

lines = Path('experiments.jsonl').read_text().strip().split('\n')
last_run = json.loads(lines[-1])

print(f'Label:           {last_run["label"]}')
print(f'Wall time:       {last_run["wall_time_s"]:.1f} s')
print()
print(f'Baseline params: {last_run["baseline"]["params_m"]:.3f} M')
print(f'Final    params: {last_run["final"]["params_m"]:.3f} M')
print(f'Param reduction: {last_run["final"]["param_reduction_pct"]:.2f} %')
print(f'MAC   reduction: {last_run["final"]["mac_reduction_pct"]:.2f} %')
print()
print(f'pose mAP50-95:   {last_run["final"]["pose_map5095"]:.4f}')
print(f'box  mAP50-95:   {last_run["final"]["box_map5095"]:.4f}')

## 4. Verify `last.pt` IS the pruned model

The most recent training run lives at `runs/pose/train<N>/weights/last.pt` (latest N). Loading it requires importing `C2f_v2` first (the surgery class is needed by the unpickler), then `YOLO()` reconstructs the pruned architecture from the saved tensors.

**Cross-check**: the loaded model's parameter count should match `experiments.jsonl`'s `final.params_m` exactly.

In [ ]:
import sys
sys.path.insert(0, '.')
from prune_yolov8_pose import C2f_v2  # required so pickle can deserialize
from ultralytics import YOLO

# Find the most recent train directory
train_dirs = sorted(Path('runs/pose').glob('train*'), key=lambda p: p.stat().st_mtime)
latest_pt = train_dirs[-1] / 'weights' / 'last.pt'
print(f'Loading: {latest_pt}')

yolo_loaded = YOLO(str(latest_pt))
loaded_params = sum(p.numel() for p in yolo_loaded.model.parameters()) / 1e6
print(f'Loaded model params: {loaded_params:.3f} M')
print(f'Match with log:      {"YES" if abs(loaded_params - last_run["final"]["params_m"]) < 0.01 else "NO"}')

# Show one representative pruned layer
model_7 = yolo_loaded.model.model[7]
if hasattr(model_7, 'conv'):
    print(f'\nmodel[7].conv weight shape: {tuple(model_7.conv.weight.shape)}')
    print('(baseline yolov8n-pose has shape (256, 128, 3, 3) — pruned should be smaller)')

## 5. Inspect the exported ONNX file

`prune_yolov8_pose_simple.py` writes `pruned_model.onnx` at the end. This file has the *real* per-tensor shapes baked into the graph definition — opens cleanly in [Netron](https://netron.app/) and reflects the actual pruned architecture.

Below we read the ONNX file's metadata and confirm a few pruned-layer shapes match what we expect.

In [ ]:
import os

onnx_path = Path('pruned_model.onnx')
if not onnx_path.exists():
    print('pruned_model.onnx not found — ONNX export may have failed on this hardware.')
    print('Check the script output for an error message; pruned_model.pt is the fallback.')
else:
    print(f'File:     {onnx_path}')
    print(f'Size:     {onnx_path.stat().st_size / 1024 / 1024:.2f} MB')
    try:
        import onnx
        model = onnx.load(str(onnx_path))
        print(f'IR ver:   {model.ir_version}')
        print(f'Opset:    {[op.version for op in model.opset_import]}')
        n_conv = sum(1 for n in model.graph.node if n.op_type == 'Conv')
        print(f'Conv ops: {n_conv}')
        # Inspect first Conv's weight shape from the initializers
        for init in model.graph.initializer[:8]:
            if 'weight' in init.name.lower():
                print(f'  {init.name}: shape={list(init.dims)}')
                break
    except ImportError:
        print('(install `onnx` to inspect — `pip install onnx`)')
    print('\nOpen in Netron with:')
    print(f'  netron {onnx_path}    # or drag-and-drop into https://netron.app/')

## 6. Cleanup (optional)

Remove the test run's logs/artifacts. The auto-log entry written by the toy run is removed from `experiments.jsonl` to keep it focused on real experiments.

In [ ]:
import shutil

# Drop the last line of experiments.jsonl (the toy run)
log_path = Path('experiments.jsonl')
lines = log_path.read_text().strip().split('\n')
log_path.write_text('\n'.join(lines[:-1]) + ('\n' if lines[:-1] else ''))
print(f'experiments.jsonl now has {len(lines) - 1} entries (dropped the toy run)')

# Optionally also wipe runs/ and the onnx file. Comment out if you want to keep them.
# shutil.rmtree('runs', ignore_errors=True)
# Path('pruned_model.onnx').unlink(missing_ok=True)
# print('runs/ and pruned_model.onnx removed')

---

## Next step — real training run

If everything above succeeded, you're ready to launch a real run. From the terminal:

```bash
python prune_yolov8_pose_simple.py \
    --data coco-pose.yaml \
    --epochs 50 --imgsz 640 --batch 16 \
    --lr 5e-5 \
    --asymmetric --asymmetric-scale 0.7
```

Wall time ~2h on RTX 5090. Target result: pose mAP50-95 ≈ 0.485 at ~14.86 % param reduction.